# CMAPSS Data Explorer (All 4 Subsets)

This notebook helps you inspect the NASA CMAPSS turbofan dataset across FD001, FD002, FD003, and FD004.

It focuses on:
- Quickly viewing the raw tables
- Understanding structure and quality
- Visualizing key patterns
- Comparing 4 focus features across all subsets

In [5]:
import os
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None
    plt.style.use("seaborn-v0_8-whitegrid")

SUBSETS = ["FD001", "FD002", "FD003", "FD004"]
COLUMN_NAMES = ["unit_id", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]
FOCUS_FEATURES = ["op1", "op2", "op3", "s7"]


def _has_cmaps_files(path: str) -> bool:
    return os.path.exists(os.path.join(path, "train_FD001.txt"))


def resolve_data_dir():
    candidates = [
        "CMAPSSData",
        os.path.join(os.getcwd(), "CMAPSSData"),
        "/Users/manankapoor/Downloads/aiproj/CMAPSSData",
        "/content/CMAPSSData",
    ]

    for c in candidates:
        if _has_cmaps_files(c):
            return c

    # Fallback: download via kagglehub if dataset files are not found locally.
    try:
        import kagglehub
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "kagglehub"])
        import kagglehub

    path = kagglehub.dataset_download("behrad3d/nasa-cmaps")
    c_maps_dir = os.path.join(path, "CMaps")

    if _has_cmaps_files(c_maps_dir):
        print(f"Downloaded CMAPSS with kagglehub: {c_maps_dir}")
        return c_maps_dir

    if _has_cmaps_files(path):
        print(f"Downloaded CMAPSS with kagglehub: {path}")
        return path

    raise FileNotFoundError(
        "CMAPSS data was not found locally and kagglehub download did not produce expected files."
    )


DATA_DIR = resolve_data_dir()
print(f"Using DATA_DIR: {os.path.abspath(DATA_DIR)}")


def load_subset(data_dir: str, subset: str):
    train_path = os.path.join(data_dir, f"train_{subset}.txt")
    test_path = os.path.join(data_dir, f"test_{subset}.txt")
    rul_path = os.path.join(data_dir, f"RUL_{subset}.txt")

    for path in [train_path, test_path, rul_path]:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing required file: {path}")

    train_df = pd.read_csv(train_path, sep=r"\s+", header=None, names=COLUMN_NAMES)
    test_df = pd.read_csv(test_path, sep=r"\s+", header=None, names=COLUMN_NAMES)
    rul_df = pd.read_csv(rul_path, sep=r"\s+", header=None, names=["rul"])
    return train_df, test_df, rul_df


datasets = {}
train_frames = []

for subset in SUBSETS:
    train_df, test_df, rul_df = load_subset(DATA_DIR, subset)

    train_df = train_df.copy()
    test_df = test_df.copy()
    rul_df = rul_df.copy()

    train_df["subset"] = subset
    test_df["subset"] = subset
    rul_df["subset"] = subset

    datasets[subset] = {
        "train": train_df,
        "test": test_df,
        "rul": rul_df,
    }
    train_frames.append(train_df)

full_train = pd.concat(train_frames, ignore_index=True)

display(Markdown("## 1) Load and Preview the Dataset"))
for subset in SUBSETS:
    display(Markdown(f"### {subset}"))
    display(Markdown("Train (first 5 rows)"))
    display(datasets[subset]["train"].head())
    display(Markdown("Train (last 5 rows)"))
    display(datasets[subset]["train"].tail())
    display(Markdown("Test (first 5 rows)"))
    display(datasets[subset]["test"].head())
    display(Markdown("RUL (first 5 rows)"))
    display(datasets[subset]["rul"].head())

100%|██████████| 12.3M/12.3M [00:00<00:00, 75.1MB/s]

Extracting files...


Downloaded CMAPSS with kagglehub: /root/.cache/kagglehub/datasets/behrad3d/nasa-cmaps/versions/1/CMaps
Using DATA_DIR: /root/.cache/kagglehub/datasets/behrad3d/nasa-cmaps/versions/1/CMaps


## 1) Load and Preview the Dataset

### FD001

...

,unit_id,cycle,op1,op2,op3,s1,s2,s3,s4,s5,...,s13,s14,s15,s16,s17,s18,s19,s20,s21,subset
61244,249,251,9.9998,0.2500,100.0,489.05,605.33,1516.36,1315.28,10.52,...,2388.73,8185.69,8.4541,0.03,372,2319,100.0,29.11,17.5234,FD004
61245,249,252,0.0028,0.0015,100.0,518.67,643.42,1598.92,1426.77,14.62,...,2388.46,8185.47,8.2221,0.03,396,2388,100.0,39.38,23.7151,FD004
61246,249,253,0.0029,0.0000,100.0,518.67,643.68,1607.72,1430.56,14.62,...,2388.48,8193.94,8.2525,0.03,395,2388,100.0,39.78,23.8270,FD004
61247,249,254,35.0046,0.8400,100.0,449.44,555.77,1381.29,1148.18,5.48,...,2388.83,8125.64,9.0515,0.02,337,2223,100.0,15.26,9.0774,FD004
61248,249,255,42.0030,0.8400,100.0,445.00,549.85,1369.75,1147.45,3.91,...,2388.66,8144.33,9.1207,0.02,333,2212,100.0,10.66,6.4341,FD004


Test (first 5 rows)

,unit_id,cycle,op1,op2,op3,s1,s2,s3,s4,s5,...,s13,s14,s15,s16,s17,s18,s19,s20,s21,subset
0,1,1,20.0072,0.7000,100.0,491.19,606.67,1481.04,1227.81,9.35,...,2387.78,8048.98,9.2229,0.02,362,2324,100.00,24.31,14.7007,FD004
1,1,2,24.9984,0.6200,60.0,462.54,536.22,1256.17,1031.48,7.05,...,2028.09,7863.46,10.8632,0.02,306,1915,84.93,14.36,8.5748,FD004
2,1,3,42.0000,0.8420,100.0,445.00,549.23,1340.13,1105.88,3.91,...,2387.95,8071.13,9.3960,0.02,328,2212,100.00,10.39,6.4365,FD004
3,1,4,42.0035,0.8402,100.0,445.00,549.19,1339.70,1107.26,3.91,...,2387.90,8078.89,9.3594,0.02,328,2212,100.00,10.56,6.2367,FD004
4,1,5,35.0079,0.8400,100.0,449.44,555.10,1353.04,1117.80,5.48,...,2387.87,8057.83,9.3030,0.02,333,2223,100.00,14.85,8.9326,FD004


RUL (first 5 rows)

,rul,subset
0,22,FD004
1,39,FD004
2,107,FD004
3,75,FD004
4,149,FD004


In [ ]:
display(Markdown("## 2) Inspect Shape, Column Types, and Basic Info"))

shape_rows = []
for subset in SUBSETS:
    train_df = datasets[subset]["train"]
    test_df = datasets[subset]["test"]

    shape_rows.append({
        "subset": subset,
        "train_rows": train_df.shape[0],
        "train_cols": train_df.shape[1],
        "test_rows": test_df.shape[0],
        "test_cols": test_df.shape[1],
        "train_engines": train_df["unit_id"].nunique(),
        "test_engines": test_df["unit_id"].nunique(),
        "max_train_cycle": train_df.groupby("unit_id")["cycle"].max().max(),
        "mean_train_cycle": round(train_df.groupby("unit_id")["cycle"].max().mean(), 2),
    })

shape_df = pd.DataFrame(shape_rows)
display(shape_df)

for subset in SUBSETS:
    display(Markdown(f"### {subset} column dtypes"))
    display(datasets[subset]["train"].dtypes.rename("dtype").to_frame())

for subset in SUBSETS:
    print(f"\n===== {subset} train info() =====")
    datasets[subset]["train"].info()

In [ ]:
display(Markdown("## 3) Display Missing Values and Data Quality Checks"))

quality_rows = []
for subset in SUBSETS:
    for split in ["train", "test", "rul"]:
        df = datasets[subset][split]
        quality_rows.append({
            "subset": subset,
            "split": split,
            "rows": len(df),
            "columns": df.shape[1],
            "total_missing_values": int(df.isna().sum().sum()),
            "duplicate_rows": int(df.duplicated().sum()),
        })

quality_df = pd.DataFrame(quality_rows)
display(quality_df)

issues_df = quality_df[(quality_df["total_missing_values"] > 0) | (quality_df["duplicate_rows"] > 0)]
if len(issues_df) == 0:
    display(Markdown("No nulls or duplicate rows detected in the loaded splits."))
else:
    display(Markdown("Potential issues found:"))
    display(issues_df)

In [ ]:
display(Markdown("## 4) Compute Summary Statistics for All 4 Features"))

stats_tables = []
for subset in SUBSETS:
    train_df = datasets[subset]["train"]
    desc = train_df[FOCUS_FEATURES].describe().T
    desc["median"] = train_df[FOCUS_FEATURES].median()
    desc = desc[["min", "25%", "50%", "median", "mean", "75%", "max", "std"]]
    desc["subset"] = subset
    desc["feature"] = desc.index
    stats_tables.append(desc.reset_index(drop=True))

stats_df = pd.concat(stats_tables, ignore_index=True)
display(stats_df[["subset", "feature", "min", "25%", "50%", "median", "mean", "75%", "max", "std"]])

agg_df = full_train.groupby("subset")[FOCUS_FEATURES].agg(["mean", "median", "std", "min", "max"])
display(Markdown("Grouped aggregate view by subset:"))
display(agg_df.round(4))

In [ ]:
display(Markdown("## 5) Visualize Distributions of All 4 Features"))

fig, axes = plt.subplots(len(FOCUS_FEATURES), 2, figsize=(14, 4 * len(FOCUS_FEATURES)))

for i, feature in enumerate(FOCUS_FEATURES):
    ax_hist = axes[i, 0]
    ax_box = axes[i, 1]

    for subset in SUBSETS:
        values = datasets[subset]["train"][feature]
        ax_hist.hist(values, bins=40, alpha=0.35, label=subset)

    ax_hist.set_title(f"Histogram: {feature}")
    ax_hist.set_xlabel(feature)
    ax_hist.set_ylabel("Count")
    if i == 0:
        ax_hist.legend()

    box_data = [datasets[s]["train"][feature].values for s in SUBSETS]
    ax_box.boxplot(box_data, labels=SUBSETS, vert=False)
    ax_box.set_title(f"Boxplot by subset: {feature}")
    ax_box.set_xlabel(feature)

plt.tight_layout()
plt.show()

In [ ]:
display(Markdown("## 6) Plot Pairwise Relationships Across the 4 Features"))

sample_for_pair = (
    full_train[["subset"] + FOCUS_FEATURES]
    .groupby("subset", group_keys=False)
    .apply(lambda df: df.sample(min(1200, len(df)), random_state=42))
    .reset_index(drop=True)
)

if sns is not None:
    sns.pairplot(
        sample_for_pair,
        vars=FOCUS_FEATURES,
        hue="subset",
        corner=True,
        plot_kws={"s": 10, "alpha": 0.5},
        diag_kws={"fill": False},
    )
    plt.show()
else:
    from pandas.plotting import scatter_matrix
    scatter_matrix(sample_for_pair[FOCUS_FEATURES], figsize=(10, 10), diagonal="hist", alpha=0.2)
    plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
pairs = [("op1", "s7"), ("op2", "op3"), ("s7", "op3")]

for ax, (x_col, y_col) in zip(axes, pairs):
    for subset in SUBSETS:
        subset_df = datasets[subset]["train"][[x_col, y_col]]
        sampled = subset_df.sample(min(1000, len(subset_df)), random_state=42)
        ax.scatter(sampled[x_col], sampled[y_col], s=10, alpha=0.35, label=subset)

    ax.set_title(f"{x_col} vs {y_col}")
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)

axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
display(Markdown("## 7) Correlation Heatmap for Numeric Features"))

for subset in SUBSETS:
    corr = datasets[subset]["train"][FOCUS_FEATURES].corr()

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)

    ax.set_xticks(range(len(FOCUS_FEATURES)))
    ax.set_yticks(range(len(FOCUS_FEATURES)))
    ax.set_xticklabels(FOCUS_FEATURES, rotation=45, ha="right")
    ax.set_yticklabels(FOCUS_FEATURES)
    ax.set_title(f"{subset} Correlation Heatmap")

    for i in range(len(FOCUS_FEATURES)):
        for j in range(len(FOCUS_FEATURES)):
            ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", color="black")

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

In [ ]:
display(Markdown("## 8) Grouped Views and Quick Comparison Charts"))

grouped_mean = full_train.groupby("subset")[FOCUS_FEATURES].mean().reset_index()
display(Markdown("Average value of each focus feature by subset:"))
display(grouped_mean)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
colors = ["#4e79a7", "#f28e2b", "#e15759", "#76b7b2"]

for ax, feature in zip(axes, FOCUS_FEATURES):
    ax.bar(grouped_mean["subset"], grouped_mean[feature], color=colors)
    ax.set_title(f"Average {feature} by subset")
    ax.set_xlabel("Subset")
    ax.set_ylabel(f"Mean {feature}")

plt.tight_layout()
plt.show()

engine_comparison = (
    full_train.groupby("subset")
    .agg(
        train_rows=("unit_id", "size"),
        unique_engines=("unit_id", "nunique"),
        max_cycle=("cycle", "max"),
    )
    .reset_index()
)

display(Markdown("Quick engine/count comparison:"))
display(engine_comparison)

## Notes

- Place your CMAPSS files in the project root folder as CMAPSSData.
- This notebook compares FD001, FD002, FD003, and FD004 side by side.
- To inspect full tables interactively, run cells and use variables like:
  - datasets["FD001"]["train"]
  - datasets["FD002"]["test"]
  - datasets["FD003"]["rul"]
- If seaborn is unavailable, the notebook still runs with matplotlib-only fallback for key visuals.